# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/03_working_with_the_full_release.ipynb)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [30]:
%pip -q install duckdb huggingface_hub


In [31]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [32]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [33]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [34]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


Grain confirmed — no duplicate (date, client, content) combinations exist. This confirms one row truly equals one client-content pair, on one day.


In [35]:
con.sql(f"""
    SELECT COUNT(*) AS rows, MIN(report_date) AS start, MAX(report_date) AS end
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

,rows,start,end
0,9841378,2026-03-01,2026-03-31


The March 2026 slice contains 9,841,378 rows, spanning the full month (2026-03-01 to 2026-03-31) with no gaps — confirming the time window stated in Section 1.

In [36]:
con.sql(f"""
    SELECT COUNT(*) AS total,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS available
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,available
0,9841378,413966


9,841,378 total, 413,966 available:

Out of 9,841,378 total rows, only 413,966 (4.2%) have ga4_data_available = TRUE. The remaining ~95.8% will show GA4-based fields (like sessions_ai) as zero or missing — but that means "not tracked yet," not "zero traffic." So sessions_ai is only counted where the flag is TRUE, to avoid a false zero-traffic signal.

In [37]:
features_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp,
           SUM(gsc_clicks) AS clicks,
           AVG(gsc_avg_position) AS position,
           SUM(gsc_clicks)*1.0/NULLIF(SUM(gsc_impressions),0) AS ctr,
           SUM(sessions_ai) FILTER (WHERE ga4_data_available IS TRUE) AS ai_sessions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1,2
    HAVING imp > 0
""").df()
print(len(features_df), "rows")
features_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151981 rows


,client_hash_id,content_hash_id,imp,clicks,position,ctr,ai_sessions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,0.001438,NaN
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,0.000000,NaN
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,0.000810,NaN
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,0.003279,NaN
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,0.0,9.000000,0.000000,NaN


After the 5-feature frame — 151,981 rows:

Five features, one line each on why they're knowable before the decision moment:

imp (total impressions, Mar 1–15) — already logged by Google Search Console before the decision point.
clicks (total clicks, Mar 1–15) — same: purely historical, pre-decision.
position (average search position, Mar 1–15) — backward-looking; today's average doesn't depend on tomorrow's ranking.
ctr (clicks/impressions) — derived only from the two historical columns above.
ai_sessions (AI-referred sessions, ga4_data_available IS TRUE only) — a real historical GA4 count; gated on the flag so "not tracked yet" isn't mistaken for "zero AI traffic."

In [38]:
label_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_label
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
    GROUP BY 1, 2
""").df()

data = features_df.merge(label_df, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["imp_label"] < 0.8 * data["imp"]).astype(int)
print(len(data), "rows | declining rate:", round(data["is_declining"].mean(), 3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151980 rows | declining rate: 0.327


151,980 rows, declining rate 0.327:

151,980 pairs had data in both the feature and label windows. 32.7% of them show impressions dropping 20%+ in the second half of March vs. the first half — this is is_declining, the target label.

In [39]:
from sklearn.tree import DecisionTreeClassifier, export_text
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

honest_features = ["imp", "clicks", "position", "ctr", "ai_sessions"]
X_honest = data[honest_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = data["is_declining"].values

honest_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_honest, y)
honest_score = honest_tree.predict_proba(X_honest)[:, 1]
print(f"HONEST Precision@50: {precision_at_k(honest_score, y, 50):.3f}")

HONEST Precision@50: 0.400


After the honest score — Precision@50 = 0.400:

Honest baseline using only the 5 features: Precision@50 = 0.400 — of the top 50 pages flagged for review, 40% actually declined. This is the number that ships.

In [40]:
# Trap: imp_label seedha label window se aaya hai — ye khud is_declining ka source hai
leaky_features = honest_features + ["imp_label"]
X_leaky = data[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)

leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
leaky_score = leaky_tree.predict_proba(X_leaky)[:, 1]
print(f"LEAKY Precision@50: {precision_at_k(leaky_score, y, 50):.3f}  <- ye dekhna, jump hoga")
print()
print(export_text(leaky_tree, feature_names=leaky_features))

LEAKY Precision@50: 1.000  <- ye dekhna, jump hoga

|--- imp_label <= 3.50
|   |--- imp_label <= 0.50
|   |   |--- class: 1
|   |--- imp_label >  0.50
|   |   |--- class: 1
|--- imp_label >  3.50
|   |--- imp <= 7.50
|   |   |--- class: 0
|   |--- imp >  7.50
|   |   |--- class: 0



In [41]:
final_features = honest_features
print("Final feature set:", final_features)
print(f"Final, honest Precision@50: {precision_at_k(honest_score, y, 50):.3f}")

Final feature set: ['imp', 'clicks', 'position', 'ctr', 'ai_sessions']
Final, honest Precision@50: 0.400


In [42]:
partial_clients = con.sql(f"""
    SELECT client_hash_id, COUNT(DISTINCT report_date) AS days_present
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1
    HAVING COUNT(DISTINCT report_date) < 15
""").df()
print(len(partial_clients), "clients had a partial feature window in March")

1 clients had a partial feature window in March


After removing the leak — Precision@50 = 0.400 again:

imp_label is removed. The final feature set is the original 5, and the honest, reported number is Precision@50 = 0.400.

After the limitation check — 1 client with partial window:

Only 1 client had a partial feature window (fewer than 15 days) in March; the rest had full coverage. Panel imbalance had limited effect on this particular slice, though it could matter more across the full 17-month dataset.

## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [43]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,247 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


In [44]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
features.head()

,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [46]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,247 rows
joined: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [47]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.633
base rate (always predict majority): 0.633
              precision    recall  f1-score   support

           0      0.547     0.336     0.416      9389
           1      0.685     0.839     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.587     0.585     25551
weighted avg      0.634     0.654     0.630     25551

              precision    recall  f1-score   support

           0      0.547     0.336     0.416      9389
           1      0.685     0.839     0.754     16162

    accuracy                          0.654     25551
   macro avg      0.616     0.587     0.585     25551
weighted avg      0.634     0.654     0.630     25551



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


Summary: ML-04 Data Contract — Lane 2 (Refresh / Content Opportunity Scoring)

1. The Contract

One row in this analysis means one (client, content item) pair, aggregated over a defined time window from fact_content_daily_performance. The table used is the warehouse fact table itself, sliced to month=2026-03 — a mid-panel month chosen specifically because the assignment warns that the final month (_sample, June 2026) is a sealed test window and must never be used to build label logic. The time window was split in two: a feature window (March 1–15) representing everything knowable at the decision point, and a label window (March 16–31) representing the future outcome being predicted. The target is a proxy: is_declining — whether a page's search impressions dropped 20%+ from the first half of March to the second half. One thing deliberately excluded from this contract was fact_content_query_90d, because its 90-day window overlaps the snapshot's later months in a way that wasn't verified here, and including it without that check risked introducing unverified assumptions.

2. Three Verification Queries

Grain check: grouped by (report_date, client, content) and filtered for duplicates — the result was empty, confirming the table's grain really is one row per client-content pair per day.
Row count and date span: the March slice contains 9,841,378 rows, spanning the complete month (2026-03-01 to 2026-03-31) with no missing days.
Availability check: of those 9,841,378 rows, only 413,966 (4.2%) had ga4_data_available = TRUE. This confirms that GA4-derived signals like ai_sessions are mostly untracked rather than genuinely zero across most of the panel — a critical distinction for correct feature construction.

3. Five Features

Built strictly from the feature window (March 1–15), so nothing from the future leaks in:

imp — summed search impressions
clicks — summed search clicks
position — average search ranking position
ctr — derived click-through rate (clicks/impressions)
ai_sessions — summed AI-referred sessions, counted only where ga4_data_available was TRUE

Each is knowable at the decision moment because all of them are aggregates of data already logged before March 15 — none depend on anything that happened afterward.

4. The Leakage Trap

An honest model using only the five features above scored Precision@50 = 0.400 — meaning 40% of the top 50 pages flagged for review actually declined. When imp_label (a column taken directly from the label window — the exact source the label was computed from) was added as a sixth "feature," the score jumped to Precision@50 = 1.000, and the decision tree stopped using any real signal, splitting purely on that one leaked column. This demonstrated concretely what leakage looks like: a feature that only exists because the outcome already happened. imp_label was then removed, and the final, reported result reverts to the honest 0.400.

5. Limitation

The dataset is an unbalanced panel — client tracking history varies in length across the full warehouse. Within this specific March slice, the impact was small: only 1 of the clients present had a partial (under 15-day) feature window. But across the full 17-month dataset, this imbalance is likely more significant, and any future work extending this window should re-check per-client coverage rather than assuming a uniform panel.